In [2]:
import sympy

# Quasistatic Poroelasticity

The governing equations are

\begin{gather}
% Solution
\vec{s}^{T} = \left(\vec{u} \quad p \quad \vec{\epsilon}_v \right) \\
% Displacement
\vec{f}(\vec{x},t) + \nabla \cdot \boldsymbol{\sigma}(\vec{u},p) = 0 \text{ in } \Omega \\
% Pressure
\frac{\partial \zeta(\vec{u},p)}{\partial t } - \gamma(\vec{x},t) + \nabla \cdot \vec{q}(p) = 0 \text{ in } \Omega \\
% Neumann traction
\boldsymbol{\sigma} \cdot \vec{n} = \vec{\tau}(\vec{x},t) \text{ on } \Gamma_{\tau} \\
% Dirichlet displacement
\vec{u} = \vec{u}_{0}(\vec{x}, t) \text{ on } \Gamma_{u} \\
% Neumann flow
\vec{q} \cdot \vec{n} = q_{0}(\vec{x}, t) \text{ on } \Gamma_{q} \\
% Dirichlet pressure
p = p_{0}(\vec{x},t) \text{ on } \Gamma_{p}
\end{gather}

where

\begin{gather}
%
  \vec{q}(p) = -\frac{\boldsymbol{k}}{\mu_{f}}(\nabla p - \vec{f}_f), \\
%
  \zeta(\vec{u},p) = \alpha (\nabla \cdot \vec{u}) + \frac{p}{M}, \\
%
  \boldsymbol{\sigma}(\vec{u},p) = \boldsymbol{C}:\boldsymbol{\epsilon} - \alpha p \boldsymbol{I}
  = \lambda \boldsymbol{I} \epsilon_{v} + 2 \mu \boldsymbol{\epsilon}  - \alpha \boldsymbol{I} p, \\
\lambda = K_{d} - \frac{2}{3} \mu, \\
  \frac{1}{M} = \frac{\alpha-\phi}{K_s} + \frac{\phi}{K_f}, \\
  \alpha = 1 - \frac{K_d}{K_f}, \\
\epsilon_{v} = \nabla \cdot \vec{u},
\end{gather}

$M$ denotes the Biot modulus, $\alpha$ denotes the Biot coefficient, $1/M$ is the specific storage coefficient at constant strain, $K_d$ denotes the bulk modulus of the drained system, $K_s$ denotes the bulk modulus of the solid, and $K_f$ denotes the bulk modulus of the fluid, $\mu$ denotes the shear modulus, and $\epsilon_{v}$ denotes the volumetric strain.

## Test Case: Linear Gradient in Fluid Pressure

We consider a linear gradient in fluid pressure along the x direction,

\begin{equation}
p(x) = p_0 \left( 1 - \frac{x}{L} \right).
\end{equation}

Solving for $\vec{q}$ with $\vec{f}_f = \vec{0}$, we have

\begin{aligned}
q_x &= \frac{k}{\mu_f} \frac{p_0}{L}\\
q_y &= 0
\end{aligned}

Solving the elasticity equation leads to

\begin{aligned}
u_x(x) &= -\frac{1}{2} \frac{\alpha p_0}{\lambda + 2\mu} \frac{x^2}{L}, \\
u_y &= 0, \\
\epsilon_v &= - \frac{\alpha p_0}{\lambda + 2\mu} \frac{x}{L}, \\
\sigma_{xx} &= -\alpha p_0, \\
\sigma_{yy} &= -\alpha p_0 \left( 1 + \frac{x}{L} \left( 1 - \frac{\lambda}{\lambda+2\mu}\right)\right), \\
\sigma_{xy} &= 0.
\end{aligned}

### Verify the analytical solution

In [3]:
# Define symbolic variable
x, y = sympy.symbols("x, y")

# Define symbolic constants
p0, L = sympy.symbols("p0, L")

# Material parameters
λ, μ, ϕ, α, μf, k, ρb, ρf, M = sympy.symbols("λ, μ, ϕ, α, μf, k, ρb, ρf, M")

# Analytical solution
ux = -0.5*α*p0 /(λ+2*μ) * x**2 / L
uy = 0 * x
p = p0 * (1 - x/L)

# Derivatives for governing equations
ux_x = ux.diff(x)
ux_y = ux.diff(y)
uy_x = uy.diff(x)
uy_y = uy.diff(y)
p_x = p.diff(x)
p_y = p.diff(y)
grad_p = sympy.Matrix([p_x, p_y])

# Body force for fluid phase (assume 0)
ff = sympy.Matrix([0., 0.])

# Darcy flux; Generalized Dacy's law
q = -(k/μf)*( grad_p + - ff)

# Strain
ϵxy = (ux_y + uy_x) / 2
ϵ = sympy.Matrix([[ux_x, ϵxy],[ϵxy, uy_y]])
ϵv = sympy.trace(ϵ)

# Stress
σ  = λ * ϵv * sympy.eye(2) + 2 * μ * ϵ - α * sympy.eye(2) * p

# Variation of fluid content
ζ = α * ϵv + p/M

# Divergence of stress
div_σ = sympy.Matrix([σ[0,0].diff(x) + σ[0,1].diff(y), σ[1,0].diff(x) + σ[1,1].diff(y)])

# Divergence of flux
div_q = q[0].diff(x) + q[1].diff(y)

In [4]:
sympy.simplify(σ)

Matrix([
[-p0*α,                                         0],
[    0, p0*α*(-L*λ - 2*L*μ + 2*x*μ)/(L*(λ + 2*μ))]])

In [5]:
sympy.simplify(ϵ)

Matrix([
[-1.0*p0*x*α/(L*(λ + 2*μ)), 0],
[                        0, 0]])

In [6]:
sympy.simplify(q)

Matrix([
[k*p0/(L*μf)],
[          0]])

In [7]:
sympy.simplify(div_σ)

Matrix([
[0],
[0]])

In [8]:
sympy.simplify(div_q)

0

## Test Case: Isotropic Linear Poro-viscoelastic rheology with a linear pressure gradient

This is a similar case to above, with the exception that we now consider the solid matrix to be a 1D linear Maxwell viscoelastic system with a time evolution goverened by

\begin{equation}
\frac{d\epsilon_T}{dt} = \frac{d\epsilon_D}{dt}+\frac{d\epsilon_S}{dt} = \frac{\sigma}{\eta_s} + \frac{1}{\mu}\frac{d\sigma}{dt}
\end{equation}

Where $\epsilon_T$ is the total strain, $\epsilon_S$ is the strain from the instantaneous poroelastic response (the spring in the Maxwell model), $\epsilon_D$ is the viscous strain (the dashpot in the Maxwell model), and $\eta_s$ is the solid viscosity.


Because the poroelastic stress and strain are not time dependant, this simplifies to
\begin{aligned}
\frac{d\epsilon_D}{dt} = \frac{\sigma}{\eta_s}
\\
\epsilon_D = \frac{\sigma}{\eta_s}t
\end{aligned}

The total deformations for this case are
\begin{aligned}
u_x(x, t) = -\frac{1}{2} \frac{\alpha p_0}{\lambda + 2\mu} \frac{x^2}{L} -\frac{\alpha p_0 x}{\eta_s}t
\\
u_y(x, y, t) =  \frac{\alpha p_0(-L\lambda-2L\mu+2x\mu)y}{L\eta_s(\lambda+2\mu)}t
\end{aligned}

To maintain a constant stress from a constant pressure gradient, we now have a non-zero fluid injection rate.
\begin{align}
\zeta(\vec{u},p) = \alpha \epsilon_{v, T} + \frac{p}{M}
\\
\frac{\partial \zeta(\vec{u},p)}{\partial t } = \gamma(\vec{x},t) = \frac{2p_0\alpha^2(-L\lambda-2L\mu+x\mu)}{L\eta_s(\lambda+2\mu)}
\end{align}
\\

To satisfy the momentum equation, we also require a body force of 
$$f_x(\vec{x}, t) = -\frac{2p_0\alpha\lambda\mu}{L\eta_s(\lambda + 2\mu)}t$$
$$f_y(\vec{x}, t) = 0$$

In [9]:
eta, t = sympy.symbols('\u03B7, t')

# Viscous strain
ϵd = (σ/eta)*t

# total strain
ϵ_tot = ϵ+ϵd
ϵv_tot = sympy.trace(ϵ_tot)


# total deformation
ux_tot = sympy.integrate(ϵ_tot[0,0], (x, 0, x))
uy_tot = sympy.integrate(ϵ_tot[1,1], (y, 0, y))

# velocities
ux_t_tot = ux_tot.diff(t)
uy_t_tot = uy_tot.diff(t)

# Variation of fluid content with total strain
ζ_tot = α * ϵv_tot + p/M
ζ_tot_t = ζ_tot.diff(t)

# Stress
σ_tot  = λ * ϵv_tot * sympy.eye(2) + 2 * μ * ϵ_tot - α * sympy.eye(2) * p

# Divergence of stress
div_σ_tot = sympy.Matrix([σ_tot[0,0].diff(x) + σ_tot[0,1].diff(y), σ_tot[1,0].diff(x) + σ_tot[1,1].diff(y)])

In [10]:
sympy.simplify(ux_tot)

1.0*p0*x*α*(-L*t*(2.0*λ + 4.0*μ) - x*η)/(L*η*(2.0*λ + 4.0*μ))

In [11]:
sympy.simplify(uy_tot)

p0*t*y*α*(-L*λ - 2*L*μ + 2*x*μ)/(L*η*(λ + 2*μ))

In [12]:
sympy.simplify(σ_tot[1,1])

p0*α*(-2*t*μ*(1.0*x*λ + (L - x)*(λ + 2*μ)) - η*(L - x)*(λ + 2*μ) - λ*(t*(1.0*x*λ + x*(1.0*λ + 2.0*μ) + 2*(L - x)*(λ + 2*μ)) + 1.0*x*η))/(L*η*(λ + 2*μ))

In [13]:
sympy.simplify(ϵv_tot)

p0*α*(-2.0*L*t*λ - 4.0*L*t*μ + 2.0*t*x*μ - 1.0*x*η)/(L*η*(1.0*λ + 2.0*μ))

In [14]:
sympy.simplify(ζ_tot_t)

p0*α**2*(-2.0*L*λ - 4.0*L*μ + 2.0*x*μ)/(L*η*(1.0*λ + 2.0*μ))

In [15]:
sympy.simplify(ux_t_tot)

-1.0*p0*x*α/η

In [16]:
sympy.simplify(uy_t_tot)

p0*y*α*(-L*λ - 2*L*μ + 2*x*μ)/(L*η*(λ + 2*μ))

In [22]:
sympy.simplify(ϵ_tot[1,1])

p0*t*α*(-L*λ - 2*L*μ + 2*x*μ)/(L*η*(λ + 2*μ))

In [18]:
sympy.simplify(div_σ_tot)

Matrix([
[2*p0*t*α*λ*μ/(L*η*(λ + 2*μ))],
[                           0]])